# LeetCode #1111 — Maximum Nesting Depth of Two Valid Parentheses Strings

**Medium · Python 3 · Stack / Greedy / Parity**  
[Đề gốc](https://leetcode.com/problems/maximum-nesting-depth-of-two-valid-parentheses-strings/) · [Mô tả và ràng buộc](https://leetcode.doocs.org/en/lc/1111/)

```python
class Solution:
    def maxDepthAfterSplit(self, seq: str) -> List[int]:
        ...
```

Mục tiêu: chia chuỗi ngoặc đúng thành hai dãy con cũng đúng sao cho **độ lồng ngoặc lớn nhất trong hai nhóm nhỏ nhất**. Notebook dạy từ đếm độ sâu tới quy tắc chia **xen kẽ theo tầng**.

## 1. Mục tiêu học

1. Hiểu ngoặc hợp lệ (valid parentheses string, VPS), **balance** và **nesting depth**.
2. Hiểu `subsequence` khác `substring`, và tại sao `answer[i]` gán ký tự vào nhóm A/B.
3. Suy ra cận dưới `ceil(D/2)` khi độ sâu ban đầu là `D`.
4. Tự nghĩ cách tham lam (*greedy*) theo **tính chẵn lẻ của độ sâu** và chứng minh tối ưu.
5. Viết test kiểm tra **tính hợp lệ + tối ưu**, không phụ thuộc duy nhất một mảng kết quả.

## 2. Đề bài, ví dụ và ràng buộc

`seq` là chuỗi chỉ chứa `(` và `)` và **đúng ngoặc**. Chia các ký tự vào hai **dãy con rời nhau** `A`, `B`, sao cho khi giữ thứ tự gốc thì cả `A` và `B` đều đúng ngoặc (có thể rỗng).

Trả mảng nhãn dài `len(seq)`: `answer[i]=0` nếu ký tự `seq[i]` thuộc `A`, `1` nếu thuộc `B`. Tối thiểu hóa `max(depth(A), depth(B))`. **Có thể có nhiều đáp án tối ưu.**

- Ví dụ 1: `seq="(()())"`; một đáp án hợp lệ: `[0,1,1,1,1,0]`.
- Ví dụ 2: `seq="()(())()"`; một đáp án hợp lệ: `[0,0,0,1,1,0,1,1]`.
- Ràng buộc: `1 <= len(seq) <= 10000`; `seq` được đảm bảo là chuỗi ngoặc đúng. Vì ngoặc đúng có độ dài chẵn, không xảy ra `len(seq)=1` trong input hợp lệ.

## 3. Nền tảng: ngoặc đúng và balance

Duyệt từ trái sang phải:
- gặp `(` thì `balance += 1` (mở thêm một cặp chưa đóng);
- gặp `)` thì `balance -= 1` (đóng một cặp);
- mọi **tiền tố** phải có `balance >= 0` và cuối chuỗi phải về `0`.

`balance` cũng là **số lớp ngoặc đang mở**. Độ lồng lớn nhất `D` là giá trị lớn nhất của `balance`.

In [1]:
def is_valid_parentheses(s):
    balance=0
    for char in s:
        if char=='(': balance+=1
        elif char==')': balance-=1
        else: return False
        if balance<0: return False
    return balance==0

for s in ['','()','(())','()(()())',')(','(()','())(']:
    print(f'{s!r:12} -> {is_valid_parentheses(s)}')
assert is_valid_parentheses('(()())')
assert not is_valid_parentheses(')(')

''           -> True
'()'         -> True
'(())'       -> True
'()(()())'   -> True
')('         -> False
'(()'        -> False
'())('       -> False


### 3.1. Đếm độ sâu mà không cần lưu cả stack

Nếu chỉ có **một loại ngoặc** thì biến số nguyên `balance` đủ. Với nhiều loại `() [] {}` cần biết **ngoặc nào** đang chờ đóng, vì thế thường cần stack để so cặp.

In [2]:
def nesting_depth(s):
    balance=best=0
    for c in s:
        if c=='(':
            balance+=1
            best=max(best,balance)
        else:
            balance-=1
        if balance<0:
            raise ValueError('Chuỗi ngoặc sai')
    if balance!=0:
        raise ValueError('Chuỗi ngoặc sai')
    return best

for s in ['','()','(())','(()())','((()))','()(())()']:
    print(f'{s!r:12} sâu={nesting_depth(s)}')

''           sâu=0
'()'         sâu=1
'(())'       sâu=2
'(()())'     sâu=2
'((()))'     sâu=3
'()(())()'   sâu=2


## 4. `subsequence` khác `substring`

- **Substring:** các ký tự phải liên tiếp trong chuỗi ban đầu.
- **Subsequence:** chỉ cần giữ **thứ tự tương đối**, có thể bỏ qua vị trí ở giữa.

Ở bài này, nếu `seq="(()())"`, phân nhóm `0/1` cho từng ký tự; gom ký tự có nhãn 0 thành `A`, còn nhãn 1 thành `B`. **Không được đổi thứ tự ký tự trong một nhóm.**

In [3]:
def split_into_groups(seq, answer):
    if len(seq)!=len(answer) or any(g not in (0,1) for g in answer):
        raise ValueError('Nhãn nhóm không hợp lệ')
    A=''.join(c for c,g in zip(seq,answer) if g==0)
    B=''.join(c for c,g in zip(seq,answer) if g==1)
    return A,B

seq='(()())'; labels=[0,1,1,1,1,0]
A,B=split_into_groups(seq,labels)
print('A =',repr(A),'B =',repr(B))
assert A=='()' and B=='()()'

A = '()' B = '()()'


## 5. Python căn bản: `enumerate`, `zip`, list comprehension

- `enumerate(seq)` trả cặp `(i,char)` để vừa biết vị trí vừa biết ký tự.
- `zip(seq,answer)` ghép ký tự với nhãn nhóm ở cùng chỉ số.
- `[0] * len(seq)` tạo mảng nhãn toàn 0.
- `x % 2` là số dư khi chia 2: 0 nếu chẵn, 1 nếu lẻ. `x & 1` có cùng tác dụng với x không âm.

In [4]:
for i,c in enumerate('()()'):
    print(i,c)
print('zip:',list(zip('()',[0,0])))
print('parity:',[(i,i%2,i&1) for i in range(6)])
print('mảng nhãn ban đầu:',[0]*4)

0 (
1 )
2 (
3 )
zip: [('(', 0), (')', 0)]
parity: [(0, 0, 0), (1, 1, 1), (2, 0, 0), (3, 1, 1), (4, 0, 0), (5, 1, 1)]
mảng nhãn ban đầu: [0, 0, 0, 0]


## 6. Cách đơn giản (baseline): dồn mọi ngoặc vào A

Nếu cho tất cả nhãn bằng `0` thì `A=seq`, `B=''`; cả hai vẫn hợp lệ. Nhưng độ sâu tối đa vẫn là `D`. Khi `D>=2`, ta có thể làm tốt hơn bằng cách san bớt độ sâu sang nhóm B.

In [5]:
def split_all_A(seq):
    return [0]*len(seq)

for seq in ['()','(()())','(((())))']:
    a,b=split_into_groups(seq,split_all_A(seq))
    print(seq,'→',repr(a),repr(b),'độ sâu cao nhất:',max(nesting_depth(a),nesting_depth(b)))

() → '()' '' độ sâu cao nhất: 1
(()()) → '(()())' '' độ sâu cao nhất: 2
(((()))) → '(((())))' '' độ sâu cao nhất: 4


## 7. Cận dưới: hai nhóm không thể cùng nông tùy ý

Giả sử độ sâu cực đại của `seq` là `D`: có một **tiền tố** chứa `D` dấu `(` nhiều hơn dấu `)`. Mỗi ký tự của tiền tố được gán vào A hoặc B, do đó **balance(A) + balance(B) = D** tại thời điểm đó. Vì A và B đều là chuỗi ngoặc đúng, balance của mỗi nhóm tại mọi tiền tố đều không âm.

Nếu lúc ấy A có `dA` dấu mở chưa đóng, B có `dB`, ta có `dA+dB=D`. Vậy `max(dA,dB) >= ceil(D/2)` (nguyên lý ngăn kéo). Suy ra độ sâu tối đa của ít nhất một nhóm cũng phải đạt ngưỡng này. **Không thuật toán nào đạt nhỏ hơn `ceil(D/2)`.**

Mục tiêu còn lại: xây một cách chia thật sự đạt cận này.

In [6]:
def theoretical_optimum(seq):
    D=nesting_depth(seq)
    return (D+1)//2  # ceil(D/2) khi D là số nguyên không âm

for s in ['()','(())','((()))','(((())))','(((()())))']:
    print(s,'D=',nesting_depth(s),'cận dưới=',theoretical_optimum(s))

() D= 1 cận dưới= 1
(()) D= 2 cận dưới= 1
((())) D= 3 cận dưới= 2
(((()))) D= 4 cận dưới= 2
(((()()))) D= 4 cận dưới= 2


## 8. Bước đột phá — chia **xen kẽ theo tầng**

Với các cặp ngoặc **lồng nhau**, gán xen kẽ tầng đầu (depth=1) cho A, tầng thứ hai cho B, tầng thứ ba cho A,...

**Quy tắc đủ để cách chia này đúng:** ta chủ động gán dấu mở và dấu đóng **cùng một cặp gốc** vào **cùng nhóm**. Đây là tính chất của **thuật toán parity**, không phải điều kiện bắt buộc cho mọi cách phân nhóm hợp lệ (ví dụ `"(())"` có thể chia thành hai dãy `"()"`, `"()"` bằng nhãn `[0,1,0,1]`).
- Khi gặp `(`: nhãn của nó là `depth` **trước khi tăng** `depth` (tầng này đang bắt đầu tại depth+1).
- Khi gặp `)`: phải **giảm** `depth` trước khi lấy nhãn, để nhãn trùng với dấu mở tương ứng.

Có thể dùng `depth % 2` (0/1). Nhóm A nhận tầng 1, 3, 5...; B nhận tầng 2, 4, 6... (theo quy ước đếm 0 trước mở).

In [7]:
def split_by_depth_parity(seq):
    answer=[]
    depth=0
    for char in seq:
        if char=='(':
            answer.append(depth%2)
            depth+=1
        else:
            depth-=1
            answer.append(depth%2)
    return answer

for s in ['()','(())','(()())','((()))','()(())()']:
    ans=split_by_depth_parity(s)
    a,b=split_into_groups(s,ans)
    print(f'{s!r:12} -> {ans} -> {a!r}, {b!r}')

'()'         -> [0, 0] -> '()', ''
'(())'       -> [0, 1, 1, 0] -> '()', '()'
'(()())'     -> [0, 1, 1, 1, 1, 0] -> '()', '()()'
'((()))'     -> [0, 1, 0, 0, 1, 0] -> '(())', '()'
'()(())()'   -> [0, 0, 0, 1, 1, 0, 0, 0] -> '()()()', '()'


## 9. Trace thủ công với `seq="(()())"`

| i | ký tự | depth trước | Quyết định | nhãn | depth sau |
|---:|:---:|---:|---|---:|---:|
| 0 | `(` | 0 | lấy chẵn 0, rồi tăng | 0 | 1 |
| 1 | `(` | 1 | lấy lẻ 1, rồi tăng | 1 | 2 |
| 2 | `)` | 2 | giảm thành 1, rồi lấy lẻ | 1 | 1 |
| 3 | `(` | 1 | lấy lẻ 1, rồi tăng | 1 | 2 |
| 4 | `)` | 2 | giảm thành 1, rồi lấy lẻ | 1 | 1 |
| 5 | `)` | 1 | giảm thành 0, rồi lấy chẵn | 0 | 0 |

Kết quả `[0,1,1,1,1,0]`, nhận `A="()"`, `B="()()"`. Cả hai nhóm đều đúng; độ sâu lớn nhất là 1, bằng `ceil(2/2)=1`.

In [8]:
def trace_parity(seq):
    depth=0; answer=[]
    print('i  c  trước  nhãn  sau  đáp_án_tạm')
    for i,c in enumerate(seq):
        before=depth
        if c=='(':
            group=depth%2
            depth+=1
        else:
            depth-=1
            group=depth%2
        answer.append(group)
        print(f'{i:<2} {c} {before:>5} {group:>5} {depth:>4}  {answer}')
    return answer

assert trace_parity('(()())')==[0,1,1,1,1,0]

i  c  trước  nhãn  sau  đáp_án_tạm
0  (     0     0    1  [0]
1  (     1     1    2  [0, 1]
2  )     2     1    1  [0, 1, 1]
3  (     1     1    2  [0, 1, 1, 1]
4  )     2     1    1  [0, 1, 1, 1, 1]
5  )     1     0    0  [0, 1, 1, 1, 1, 0]


## 10. Vì sao tham lam theo parity luôn đúng và tối ưu?

**Đúng ngoặc:** Mỗi dấu mở có độ sâu `d` trước lúc tăng. Khi gặp dấu đóng **khớp** với nó, depth từ `d+1` giảm về `d`, nên hai dấu đều được gán `d%2`. Vì thế các cặp không bị tách, thứ tự được giữ và từng nhóm luôn là VPS.

**Tối ưu:** Trong một chuỗi các cặp lồng nhau có độ sâu `D`, nhóm 0 chiếm tối đa `ceil(D/2)` tầng, nhóm 1 chiếm tối đa `floor(D/2)` tầng. Vậy `max(depth(A),depth(B))=ceil(D/2)`, đạt **cận dưới** ở phần 7. Do đó cách chia là tối ưu toàn cục.

## 11. Code nộp LeetCode

Không cần stack ký tự; chỉ một biến `depth` đếm mức chưa đóng. Đây là `O(n)` thời gian. Mảng kết quả O(n) là bắt buộc vì đề yêu cầu xuất n nhãn.

In [9]:
from typing import List
class Solution:
    def maxDepthAfterSplit(self, seq: str) -> List[int]:
        answer=[]
        depth=0
        for char in seq:
            if char=='(':
                answer.append(depth%2)
                depth+=1
            else:
                depth-=1
                answer.append(depth%2)
        return answer

print('Ví dụ 1:',Solution().maxDepthAfterSplit('(()())'))
print('Ví dụ 2:',Solution().maxDepthAfterSplit('()(())()'))

Ví dụ 1: [0, 1, 1, 1, 1, 0]
Ví dụ 2: [0, 0, 0, 1, 1, 0, 0, 0]


## 12. Độ phức tạp và so sánh

Gọi `n=len(seq)`, `D` là độ sâu ngoặc tối đa.

| Cách | Thời gian | Bộ nhớ phụ | Mức độ sâu đạt được |
|---|---|---|---|
| Dồn hết vào A | O(n) | O(n) kết quả | D (không tối ưu nếu D≥2) |
| **Xen kẽ theo tầng** | **O(n)** | **O(n) kết quả, O(1) biến phụ** | **ceil(D/2)** |
| Thử mọi cách gán 0/1 | O(n·2ⁿ) | O(n) | Tối ưu nhưng chỉ hợp input rất nhỏ |

Không được gọi toàn thuật toán là O(1) bộ nhớ: **mảng `answer` có n phần tử**, nên tổng bộ nhớ là O(n).

## 13. Kiểm thử đúng cách khi có nhiều đáp án tối ưu

**Sai lầm kiểm thử:** luôn yêu cầu kết quả **bằng hệt** ví dụ đề. Vì đề cho phép nhiều cách chia tối ưu, hãy kiểm tra đồng thời:
1. `len(answer)==len(seq)`, nhãn chỉ là 0 hoặc 1;
2. `A`, `B` đều hợp lệ;
3. `max(depth(A),depth(B)) == ceil(depth(seq)/2)`.

In [10]:
def check_optimal_split(seq, answer):
    if len(answer)!=len(seq) or any(group not in (0,1) for group in answer):
        return False
    A,B=split_into_groups(seq,answer)
    if not (is_valid_parentheses(A) and is_valid_parentheses(B)):
        return False
    return max(nesting_depth(A),nesting_depth(B))==theoretical_optimum(seq)

cases=['()','(())','(()())','((()))','()(())()','()()()','(((())))','(()(()))']
for s in cases:
    ans=Solution().maxDepthAfterSplit(s)
    assert check_optimal_split(s,ans),(s,ans)
    print(f'{s!r:<15} PASS; max_depth={theoretical_optimum(s)}')

'()'            PASS; max_depth=1
'(())'          PASS; max_depth=1
'(()())'        PASS; max_depth=1
'((()))'        PASS; max_depth=2
'()(())()'      PASS; max_depth=1
'()()()'        PASS; max_depth=1
'(((())))'      PASS; max_depth=2
'(()(()))'      PASS; max_depth=2


## 14. Oracle vét cạn cho input nhỏ

Với `n` ký tự, thử tất cả `2^n` cách gán từng ký tự vào A/B; bỏ cách tạo dãy ngoặc sai; lấy độ sâu tốt nhất còn lại. Vét cạn không dùng để nộp vì với n=10000 là không thể thực hiện, nhưng là **bộ kiểm chứng độc lập** rất hữu ích với chuỗi ngắn.

In [11]:
from itertools import product
def brute_minimum_depth(seq):
    best=float('inf')
    for assignment in product((0,1),repeat=len(seq)):
        A,B=split_into_groups(seq,assignment)
        if is_valid_parentheses(A) and is_valid_parentheses(B):
            best=min(best,max(nesting_depth(A),nesting_depth(B)))
    return best

for seq in ['()','(())','(()())','((()))','()(())()']:
    got=Solution().maxDepthAfterSplit(seq)
    assert check_optimal_split(seq,got)
    assert max(nesting_depth(v) for v in split_into_groups(seq,got))==brute_minimum_depth(seq)
print('PASS so sánh với vét cạn')

PASS so sánh với vét cạn


### 14.1. Sinh ngẫu nhiên các chuỗi VPS (có seed)

Chỉ mở `(` nếu còn cặp chưa mở; chỉ đóng `)` nếu có cặp đang mở. Nhờ đó chuỗi sinh ra chắc chắn hợp lệ, không cần loại các chuỗi sai. So cả với oracle cho chuỗi tới 10 ký tự.

In [12]:
import random
rng=random.Random(1111)
def random_vps(n_pairs):
    opened=closed=0; out=[]
    while closed<n_pairs:
        choices=[]
        if opened<n_pairs: choices.append('(')
        if closed<opened: choices.append(')')
        c=rng.choice(choices)
        out.append(c)
        if c=='(': opened+=1
        else: closed+=1
    return ''.join(out)

for pairs in range(1,6):
    for _ in range(40):
        seq=random_vps(pairs)
        ans=Solution().maxDepthAfterSplit(seq)
        assert check_optimal_split(seq,ans),(seq,ans)
        assert brute_minimum_depth(seq)==theoretical_optimum(seq)
print('PASS 200 VPS ngẫu nhiên và kiểm chứng vét cạn')

PASS 200 VPS ngẫu nhiên và kiểm chứng vét cạn


### 14.2. Test độ dài tối đa

`seq='('*5000+')'*5000` có đúng 10000 ký tự, độ sâu 5000. Cách chia parity phải cho độ sâu lớn nhất bằng 2500 mà không bị tràn stack.

In [13]:
large='('*5000+')'*5000
arr=Solution().maxDepthAfterSplit(large)
assert len(arr)==10000
assert check_optimal_split(large,arr)
print('PASS n=10000; độ sâu mỗi nhóm tối đa=',theoretical_optimum(large))

PASS n=10000; độ sâu mỗi nhóm tối đa= 2500


## 15. Phản ví dụ của các cách chia sai

- **Lấy parity của chỉ số `i` thay vì độ sâu:** không bảo đảm mỗi nhóm có balance hợp lệ; ví dụ `seq="()()"` gán `0,1,0,1` → `A="(("`, `B="))"` (sai).
- **Ở dấu đóng lấy parity trước khi giảm:** dấu đóng có thể vào nhóm khác dấu mở tương ứng, tạo chuỗi sai.
- **Gán tất cả vào A:** đúng VPS nhưng không tối ưu nếu `D>=2`.
- **So hệt output đề:** loại bỏ nhiều lời giải hợp lệ.

In [14]:
wrong_index=[i%2 for i in range(len('()()'))]
print('Sai theo index:',wrong_index,'→',split_into_groups('()()',wrong_index))
assert not check_optimal_split('()()',wrong_index)

def wrong_close_before_decrement(seq):
    depth=0; ans=[]
    for c in seq:
        if c=='(':
            ans.append(depth%2); depth+=1
        else:
            ans.append(depth%2); depth-=1  # SAI thứ tự.
    return ans
assert not check_optimal_split('(()())',wrong_close_before_decrement('(()())'))
print('Phát hiện được hai lỗi gán nhóm phổ biến')

Sai theo index: [0, 1, 0, 1] → ('((', '))')
Phát hiện được hai lỗi gán nhóm phổ biến


## 16. Biến thể và nhận diện pattern

- Bài yêu cầu **chia đối tượng thành nhiều nhóm để cân bằng chiều sâu** → thử xen kẽ theo độ sâu hiện tại.
- Trong thuật toán **xen kẽ theo tầng**, hai dấu khớp cùng nhóm → cần phân biệt thời điểm cập nhật depth ở mở/đóng.
- Nếu có **k** nhóm thay vì 2: có thể gán một cặp theo `depth_before_open % k`; khi đóng, giảm depth trước rồi gán. Mức sâu lớn nhất đạt `ceil(D/k)`.
- Nếu muốn trả về chính hai chuỗi thay vì mảng nhãn, dùng `split_into_groups`.

In [15]:
def split_into_k(seq,k):
    depth=0; labels=[]
    for c in seq:
        if c=='(':
            labels.append(depth%k);depth+=1
        else:
            depth-=1;labels.append(depth%k)
    return labels

def group_strings(seq,labels,k):
    return [''.join(c for c,g in zip(seq,labels) if g==i) for i in range(k)]

s='((((((()))))))'; groups=group_strings(s,split_into_k(s,3),3)
print(groups,[nesting_depth(g) for g in groups])
assert all(is_valid_parentheses(g) for g in groups)
assert max(nesting_depth(g) for g in groups)==(nesting_depth(s)+2)//3

['((()))', '(())', '(())'] [3, 2, 2]


## 17. Bài tập từ dễ đến khó

1. **Dự đoán:** Độ sâu của `"()(())"`, `"((()))"`, `"()()"` lần lượt là bao nhiêu?
2. **Tính tay:** Với `"((()))"`, tạo một mảng nhãn tối ưu và viết hai chuỗi A/B.
3. **TODO:** viết `exercise_depth(seq)` trả về độ sâu lớn nhất, giả sử seq đúng ngoặc.
4. **TODO:** viết `exercise_split(seq)` chia 2 nhóm tối ưu bằng parity.
5. **Biến thể:** viết `exercise_check(seq,assignment)` kiểm tra hai nhóm đều VPS **và** độ sâu tối ưu.

In [16]:
def exercise_depth(seq):
    # TODO: đếm balance, cập nhật maximum khi gặp '('.
    pass

def exercise_split(seq):
    # TODO: khi '(' -> gán depth%2 rồi tăng;
    #       khi ')' -> giảm trước rồi gán depth%2.
    pass

def exercise_check(seq,assignment):
    # TODO: kiểm tra length/labels, dựng hai chuỗi và so cận dưới.
    pass

# Tự làm xong mới bỏ comment:
# assert exercise_depth('((()))')==3
# assert check_optimal_split('(()())',exercise_split('(()())'))
# assert exercise_check('()()',[0,0,1,1])

## 18. Gợi ý ba mức

| Bài | Gợi ý 1: ý tưởng | Gợi ý 2: cú pháp | Gợi ý 3: gần-code |
|---|---|---|---|
| 3 | Số dấu `(` chưa được đóng | `depth += 1` | `maximum=max(maximum,depth)` ngay sau khi mở |
| 4 | Gán xen kẽ theo tầng | `% 2` | Dấu đóng phải `depth -= 1` trước khi gán |
| 5 | Đúng + tối ưu là hai điều kiện độc lập | `zip`, `is_valid_parentheses` | `max(depthA,depthB) == (D+1)//2` |

## 19. Đáp án tham khảo (ở cuối, không xem trước)

**Bài 1:** độ sâu lần lượt là `2`, `3`, `1`.  
**Bài 2:** một đáp án là `[0,1,0,0,1,0]` tạo `A="(())"`, `B="()"`; độ sâu lớn nhất 2 = ceil(3/2). (Bạn cũng có thể đảo 0 và 1.)

In [17]:
def answer_depth(seq):
    balance=best=0
    for c in seq:
        if c=='(':
            balance+=1;best=max(best,balance)
        else: balance-=1
    return best

def answer_split(seq):
    return split_by_depth_parity(seq)

def answer_check(seq,assignment):
    return check_optimal_split(seq,assignment)

assert answer_depth('((()))')==3
assert answer_check('(()())',answer_split('(()())'))
assert answer_check('()()',[0,0,1,1])
assert answer_check('((()))',[0,1,0,0,1,0])
print('Đáp án tham khảo: PASS')

Đáp án tham khảo: PASS


## 20. Checklist submit

- [ ] Tôi hiểu `answer[i]` là nhãn **của từng ký tự**, không phải số nhóm.
- [ ] Tôi biết **thuật toán này** giữ dấu mở/đóng khớp cùng nhóm để bảo đảm đúng (không phải điều kiện cần cho mọi đáp án).
- [ ] Khi gặp `(`, tính nhãn **trước** khi tăng depth.
- [ ] Khi gặp `)`, giảm depth **trước** khi tính nhãn.
- [ ] Tôi kiểm tra tính hợp lệ và cận dưới, không bắt buộc output giống mẫu.
- [ ] Tôi biết bộ nhớ **O(n)** do mảng kết quả.

**Template nhớ nhanh:** `(` → `append(depth%2); depth+=1`; `)` → `depth-=1; append(depth%2)`.